In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

data_path = next((path for root in [Path.cwd(), *Path.cwd().parents]
                  for path in [root / 'data-analytics/data/raw/Referrals.csv',
                               root / 'data/raw/Referrals.csv']
                  if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Place Referrals.csv in data-analytics/data/raw/')

ref = pd.read_csv(data_path)
print("Shape:", ref.shape)
print("Columns:", ref.columns.tolist())

## 1. Structure & Data types

In [ ]:
print("dtypes (as loaded):")
print(ref.dtypes)
print()

info = pd.DataFrame({
    'non_null': ref.notna().sum(),
    'nulls': ref.isna().sum(),
    'null_pct': (ref.isna().mean() * 100).round(1),
    'n_unique': ref.nunique(),
})
print(info)

**Observation:** no nulls anywhere; every referral has a referral_id, a risk_assessment_id, a type, urgency, status and both dates populated. All columns load as strings/objects.

## 2. Key integrity & duplicate checks

Checking `referral_id` as the primary key, and `risk_assessment_id` as the foreign key
back to whatever risk-assessment / screening produced the referral.

In [ ]:
print("DUPLICATE CHECKS")
print("Duplicate referral_id rows:", ref['referral_id'].duplicated().sum())
print("Fully duplicated rows:", ref.duplicated().sum())
print()

print("ID FORMAT CHECKS")
bad_ref_id = ref[~ref['referral_id'].str.match(r'^REF-\d{3}$')]
bad_risk_id = ref[~ref['risk_assessment_id'].str.match(r'^RSK-\d{4}$')]
print("referral_id not matching REF-###:", len(bad_ref_id))
print("risk_assessment_id not matching RSK-####:", len(bad_risk_id))
print()

print("risk_assessment_id duplicated (same screening/assessment linked to >1 referral)?")
dup_risk = ref[ref['risk_assessment_id'].duplicated(keep=False)]
print(dup_risk[['referral_id', 'risk_assessment_id']] if len(dup_risk)
      else "None - each referral maps to a unique risk_assessment_id (clean 1:1 relationship).")

**Observation:** `referral_id` is a clean primary key (unique, consistent `REF-###`
format, no duplicate rows). `risk_assessment_id` is unique per referral too, i.e. the
relationship from Referrals to whatever produced them (risk assessments) is 1:1, not 1:many.

## 3. Categorical exploration, referral type, urgency, status
No dedicated 'referral reason' field, the closest proxy to it is 'referral_type' which describes who the person was referred to, not why.

In [ ]:
print("=== referral_type value counts ===")
print(ref['referral_type'].value_counts())
print()
print("=== urgency value counts ===")
print(ref['urgency'].value_counts())
print()
print("=== status value counts ===")
print(ref['status'].value_counts())
print()
print("=== urgency x status crosstab ===")
print(pd.crosstab(ref['urgency'], ref['status']))
print()
print("=== referral_type x status crosstab ===")
print(pd.crosstab(ref['referral_type'], ref['status']))

**Observations:**

- `referral_type` has 4 distinct values: general practitioner (7), dietitian (2),
  counsellor (2), fitness coach (1). General practitioner dominates the sample.
- `urgency` ther is only one `urgent` case from 12 cases.
- `status` has 3 stages: `completed` (8), `scheduled` (2), `issued` (2), this looks like
  a referral lifecycle (`issued` → `scheduled` → `completed`), useful for funnel
  reporting, but there is no explicit `cancelled` / `declined` / `no-show` status in this
  sample, so we can't confirm whether the schema supports those terminal states.

## 4. Temporal analysis; referred_at, due_date

Parsing the ISO-8601 timestamps and checking the SLA window (due_date - referred_at) for internal consistency.

In [ ]:
for c in ['referred_at', 'due_date']:
    ref[c] = pd.to_datetime(ref[c], utc=True, errors='coerce')

print("Any unparseable dates after conversion?")
print(ref[['referred_at', 'due_date']].isna().sum())
print()

ref['lead_time_days'] = (ref['due_date'] - ref['referred_at']).dt.total_seconds() / 86400
print("Lead time (days) by urgency:")
print(ref.groupby('urgency')['lead_time_days'].agg(['min', 'max', 'mean']))
print()
print("Referrals where due_date <= referred_at (invalid / zero window):")
bad_window = ref[ref['lead_time_days'] <= 0]
print(bad_window if len(bad_window) else "None found.")

**Observations:**

- Both date fields parse cleanly as UTC timestamps, no malformed dates.
- The SLA window is perfectly consistent with urgency: `priority` referrals always get a
  **14-day** window, `urgent` referrals get a **1-day** window. This is a reliable,
  rule-derived field that can be used directly for SLA/turnaround analytics.
- No invalid (zero or negative) date windows.
- All `referred_at` values fall in a narrow 3-day band (19–21 June 2026), consistent
  with a seed dataset rather than a live production extract.

## 5. Relationship to employees, screenings & wellness activities

Q: **can a referral be traced back to the person and the wellness activity/screening that triggered it?**

In [ ]:
print("Columns present in Referrals.csv:")
print(list(ref.columns))
print()

identity_like = [c for c in ref.columns if any(k in c.lower() for k in
                 ['employee', 'participant', 'user', 'staff', 'name', 'screening',
                  'activity', 'wellness', 'reason', 'note', 'provider',
                  'organisation', 'organization'])]
print("Columns that look like employee / screening / reason / provider identifiers:")
print(identity_like if identity_like else "NONE FOUND")

**Finding:**

- The **only** relational key in `Referrals.csv` is `risk_assessment_id`. There is
  **no `employee_id` / `participant_id`** column, so a referral cannot be attributed to a
  specific person without joining to a separate Risk Assessments table that presumably links `risk_assessment_id → employee_id`.
- There is **no direct link to a wellness activity or screening_id** either. Again this
  would have to be inferred by joining through the risk assessment record.
- There is **no free-text reason/diagnosis field**.  `referral_type` tells you the
  destination (GP, dietitian, counsellor, fitness coach) but not the clinical/wellness
  reason for the referral.
- There is **no assigned-provider / organisation name**, and **no completion date**
  separate from `status`, so once a referral is `completed` we don't know exactly when
  the referred appointment/service was delivered (only when the *follow-up record* about
  it was logged. (see the Follow-Ups notebook)).

### 6. Summary

**What works well:**
- Clean primary key (`referral_id`), no duplicates, no missing values.
- `risk_assessment_id` is a well-formed, unique foreign key (1:1 with referrals).
- `urgency` and the `referred_at`/`due_date` pair form a reliable, internally consistent
  SLA model (`priority` = 14 days, `urgent` = 1 day), good for turnaround-time and
  SLA-compliance reporting.
- `status` gives a usable lifecycle field (`issued` → `scheduled` → `completed`) for
  funnel/volume reporting by `referral_type`, `urgency`, and `status`.

**Gaps / issues to flag to the platform team before this can support full referral
analytics:**
1. **No employee/participant identifier**: referrals cannot be attributed to a person,
   demographic group, or organisational unit without an external join on
   `risk_assessment_id` (the Risk Assessments source table is not part of this dataset).
2. **No screening / wellness-activity linkage**: can't answer "which screening types
 generate the most referrals" directly from this file.
3. **No referral reason/diagnosis field**: `referral_type` only captures the referral
   destination, not the underlying wellness/risk driver.
4. **No provider/organisation name** and **no distinct completion timestamp**: the
   `status` field alone can't tell us when a referral was actually resolved; that has to
   come from `Follow_Ups.csv`.
5. Only 3 status values were observed (`completed`, `scheduled`, `issued`); with only 12
   rows we cannot confirm whether other lifecycle states (e.g. `cancelled`, `declined`,
   `expired`) exist in the schema, worth checking against the platform's data model /
   system design docs rather than inferring from this small sample alone.
6. Sample size is very small (n=12); sufficient to validate structure and logic, but not
   representative for statistical/trend reporting; production volumes should be assessed
   separately.
